<a href="https://colab.research.google.com/github/FarabiOnAMission/Machine-Learning-Stuffs/blob/main/Vectors_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
class Vector:
  def __init__(self, components):
    self.components = list(components)
    self.dim = len(self.components)

  def __add__(self, other):
    return Vector(x + y for x,y in zip(self.components, other.components))

  def __sub__(self, other):
    return Vector(x - y for x,y in zip(self.components, other.components))

  def dot(self, other):
    return sum(x * y for x,y in zip(self.components, other.components))

  def magnitude(self,other):
    return sum(x**2 for x in self.components)**0.5

  def normalize(self):
    mag = self.magnitude()
    return Vector(x/mag for x in self.components)

  def cosine_similarity(self,other):
    return self.dot(other)/(self.magnitude()*other.magnitude())

  def __repr__(self):
    return f"Vector({self.components})"

In [4]:
a = Vector([1,2,3])
b = Vector([4,5,6])

print(f"a + b = {a+b}")
print(f"a - b = {a-b}")
print(f"a dot b = {a.dot(b)}")

a + b = Vector([5, 7, 9])
a - b = Vector([-3, -3, -3])
a dot b = 32


In [5]:
class Matrix:
    def __init__(self, rows):
        self.rows = [list(row) for row in rows ]
        self.shape = (len(self.rows), len(self.rows[0]))
    def __matmul__(self, other):
          if isinstance(other, Vector):
            return Vector([sum(self.rows[i][j] for j in range(self.shape[1]))
              for i in range(self.shape[0])])
          rows = []

          for i in range(self.shape[0]):
            row = []
            for j in range(other.shape[1]):
              row.append(sum(self.rows[i][k] * other.rows[k][j] for k in range(other.shape[1])
              ))
            rows.append(row)
          return Matrix(rows)
    def transpose(self):
      return Matrix([
          [self.rows[j][i] for j in range(self.shape[0])]
          for i in range(self.shape[1])
      ])

    def __repr__(self):
      return f"Matrix({self.rows})"



In [6]:

rotation_90 = Matrix([[0, -1], [1, 0]])
point = Vector([3, 1])

rotated = rotation_90 @ point
print(f"Original: {point}")
print(f"Rotated 90°: {rotated}")

Original: Vector([3, 1])
Rotated 90°: Vector([-1, 1])


In [7]:
def _is_linearly_independent(vectors):
  #vectors = [v1,v2,v3]
  # v1 = [x1,x2,x3]
  n = len(vectors)
  #counts how many vectors in the matrix
  dim = len(vectors[0].components)
  #features of each vector
  rows = Matrix([v.components[:] for v in vectors])

  """
   [
      [x1,x2,x3],
      [y1,y2,y3],
      [z1,z2,z3]
  ]
  """
  rank = 0
  #current rank

  #its a Row Matrix meaning each row contains a vector
  #Because row is easily accessible in python instead of columns

  for col in range(dim):
    #loops through each column
    pivot = None
    #looping through all rows of the current column
    for row in range(rank, len(rows)):
      if abs(rows[row][col]) > 1e-10:
        #finds the first non-zero row
        #then marks it as pivot and breaks
        pivot = row
        break
    if pivot is None:
      continue

    rows[rank], rows[pivot] = rows[pivot], rows[rank]
    #swaps the found pivot row with the rank row

    scale = rows[rank][col]

    #To make the pivot element 1 we divide the whole row by this element

    rows[rank] = [x/scale for x in rows[rank]]
    for row in range(len(rows)):
      if row != rank and abs(rows[row][col]) >1e-10:
        factor = rows[row][col]
        rows[row] = [rows[row][j] - factor * rows[rank][j] for j in range(dim)]
      #subtracts each element in the pivot column and makes it zero so theres only one row
    rank += 1

  return rank == n


def project(a,b):
  scalar = a.dot(b)/b.dot(b)
  return Vector([scalar * x for x in b.components])


def grahm_schimdt(vectors):
  orthonormal = []
  for v in vectors:
    w = v
    for u in orthonormal:
      w = w - project(v,u)
      orthonormal.append(w)
  return orthonormal